In [2]:
pip install pm4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 17.9 MB/s  0:00:00


In [3]:
import os
import pandas as pd
import pm4py

# ==========================================
# 1. DATA INGESTION & PARSING
# ==========================================
file_path = 'MiniProjet1_SOC_Authentification.csv'

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Critical Error: Cannot locate {file_path}. Ensure it is in the working directory.")

# Load the raw event log
print("[*] Loading raw SOC data...")
df = pd.read_csv(file_path)

# Force rigorous chronological parsing.
# 'mixed' format handles inconsistencies in SOC system timestamp generation.
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', dayfirst=True)

# Process Mining fundamentally relies on chronological order per case.
df = df.sort_values(by=['incident_id', 'timestamp'])

# ==========================================
# 2. DATA AUDIT & CLEANSING
# ==========================================
print("\n[*] Running Data Quality Audit...")

# Detect and patch missing values (specifically in the 'resource' column)
missing_resources = df['resource'].isnull().sum()
if missing_resources > 0:
    print(f"[!] Found {missing_resources} missing resource values. Patching with 'Unknown_System'...")
    df['resource'] = df['resource'].fillna('Unknown_System')

# Calculate core metrics
total_events = len(df)
total_cases = df['incident_id'].nunique()
total_activities = df['activity'].nunique()

print(f"    - Total Events (Log Size): {total_events}")
print(f"    - Total Cases (Incidents): {total_cases}")
print(f"    - Distinct Activities: {total_activities}")

# ==========================================
# 3. VARIANT ANALYSIS (THE REALITY CHECK)
# ==========================================
print("\n[*] Extracting Process Variants...")
# Group activities sequentially by incident to map the exact execution paths
cases = df.groupby('incident_id')['activity'].apply(tuple)
variants = cases.value_counts()

print(f"    - Total Unique Execution Paths (Variants): {len(variants)}")
print("\n[*] Top 5 Most Frequent Variants:")
for i, (variant_path, count) in enumerate(variants.head(5).items(), 1):
    path_string = ' -> '.join(variant_path)
    print(f"    Variant {i} ({count} cases): {path_string}")

# ==========================================
# 4. PM4PY STANDARDIZATION & XES EXPORT
# ==========================================
print("\n[*] Formatting for Process Mining (XES Standard)...")

# Duplicate to preserve the original dataframe for any manual Pandas analysis later
df_pm4py = df.copy()

# Remap columns to XES standard extensions required by PM4Py algorithms
df_pm4py.rename(columns={
    'incident_id': 'case:concept:name',  # The Case ID trace
    'activity': 'concept:name',          # The Action
    'timestamp': 'time:timestamp',       # The Ordering parameter
    'resource': 'org:resource'           # The Organizational Actor
}, inplace=True)

# Cast the dataframe to a formal Event Log object
event_log = pm4py.format_dataframe(
    df_pm4py,
    case_id='case:concept:name',
    activity_key='concept:name',
    timestamp_key='time:timestamp'
)

# Export the baseline artifact
output_file = 'MiniProjet1_SOC_EventLog.xes'
pm4py.write_xes(event_log, output_file)

print(f"\n[SUCCESS] Phase 1 Complete. Formatted event log saved as: {output_file}")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




[*] Loading raw SOC data...

[*] Running Data Quality Audit...
[!] Found 1 missing resource values. Patching with 'Unknown_System'...
    - Total Events (Log Size): 821
    - Total Cases (Incidents): 140
    - Distinct Activities: 11

[*] Extracting Process Variants...
    - Total Unique Execution Paths (Variants): 5

[*] Top 5 Most Frequent Variants:
    Variant 1 (73 cases): Multiple_Login_Failures -> SIEM_Alert_Created -> SOC_Triage -> IP_Investigated -> IP_Blocked -> Account_Disabled -> Password_Reset -> Incident_Closed
    Variant 2 (57 cases): Login_Failed -> Login_Success -> Session_Closed
    Variant 3 (5 cases): Multiple_Login_Failures -> SIEM_Alert_Created -> SOC_Triage -> Account_Disabled -> Incident_Closed
    Variant 4 (4 cases): Multiple_Login_Failures -> SIEM_Alert_Created -> SOC_Triage -> SOC_Triage -> IP_Blocked -> Account_Disabled -> Password_Reset -> Incident_Closed
    Variant 5 (1 cases): Multiple_Login_Failures -> SIEM_Alert_Created -> SIEM_Alert_Created -> SOC_Tr

/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


exporting log, completed traces ::   0%|          | 0/140 [00:00<?, ?it/s]


[SUCCESS] Phase 1 Complete. Formatted event log saved as: MiniProjet1_SOC_EventLog.xes


In [4]:
import os
import pm4py

# 1. Locate the reference model provided in your coursework files
reference_model_path = 'MiniProjet1_Modele_Reference_Reponse_Incident.pnml'

if os.path.exists(reference_model_path):
    print(f"[*] Found reference model: {reference_model_path}")

    # Import the Petri Net model using PM4Py
    net, initial_marking, final_marking = pm4py.read_pnml(reference_model_path)

    print(f"    - Petri Net Places: {len(net.places)}")
    print(f"    - Petri Net Transitions: {len(net.transitions)}")
    print(f"    - Petri Net Arcs: {len(net.arcs)}")
else:
    print(f"[!] Warning: '{reference_model_path}' not found in current directory. Please ensure it is uploaded if you plan to use it for Phase 4 conformance checking.")
    print(f"Files currently in directory: {os.listdir('.')}")

[*] Found reference model: MiniProjet1_Modele_Reference_Reponse_Incident.pnml
    - Petri Net Places: 9
    - Petri Net Transitions: 8
    - Petri Net Arcs: 16


In [6]:
import pm4py

# 1. Load the XES event log created in Phase 1
log_path = 'MiniProjet1_SOC_EventLog.xes'
print(f"[*] Loading event log from {log_path}...")
log = pm4py.read_xes(log_path)

# 2. Discover the Process Model as a Petri Net directly
print("[*] Running Inductive Miner to discover actual process flow...")
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(log)

# 3. Print discovered model statistics
print("[*] Process Discovery Complete!")
print(f"    - Discovered Petri Net Places: {len(net.places)}")
print(f"    - Discovered Petri Net Transitions: {len(net.transitions)}")

# 4. Generate and save Directly-Follows Graph (DFG) metrics
dfg, start_activities, end_activities = pm4py.discover_dfg(log)
print(f"    - DFG Edges Discovered: {len(dfg)}")

# 5. Save the discovered Petri net as a PNML file for Phase 4 Conformance Checking
output_model_path = 'MiniProjet1_Discovered_Model.pnml'
pm4py.write_pnml(net, initial_marking, final_marking, output_model_path)
print(f"[SUCCESS] Discovered model saved as '{output_model_path}'")

[*] Loading event log from MiniProjet1_SOC_EventLog.xes...


parsing log, completed traces ::   0%|          | 0/140 [00:00<?, ?it/s]

[*] Running Inductive Miner to discover actual process flow...
[*] Process Discovery Complete!
    - Discovered Petri Net Places: 15
    - Discovered Petri Net Transitions: 20
    - DFG Edges Discovered: 14
[SUCCESS] Discovered model saved as 'MiniProjet1_Discovered_Model.pnml'


In [8]:
import pm4py

# 1. Load the event log and the reference Petri net model
log_path = 'MiniProjet1_SOC_EventLog.xes'
ref_pnml_path = 'MiniProjet1_Modele_Reference_Reponse_Incident.pnml'

print("[*] Loading event log and reference model for conformance audit...")
log = pm4py.read_xes(log_path)
net, initial_marking, final_marking = pm4py.read_pnml(ref_pnml_path)

# 2. Calculate Token-Based Replay Fitness
print("[*] Calculating Token-Based Replay Fitness against the reference model...")
fitness = pm4py.conformance_token_based_replay(log, net, initial_marking, final_marking)
print(f"    - Token Fitness Results: {fitness}")

# 3. Calculate Precision (ETConformance / Alignments-based precision)
print("[*] Calculating Model Precision...")
precision = pm4py.precision_alignments(log, net, initial_marking, final_marking)
print(f"    - Precision Score: {precision:.4f}")

print("\n[SUCCESS] Phase 4 Conformance Audit metrics generated successfully.")

[*] Loading event log and reference model for conformance audit...


parsing log, completed traces ::   0%|          | 0/140 [00:00<?, ?it/s]

[*] Calculating Token-Based Replay Fitness against the reference model...


replaying log with TBR, completed traces ::   0%|          | 0/5 [00:00<?, ?it/s]

    - Token Fitness Results: {'perc_fit_traces': 52.142857142857146, 'average_trace_fitness': 0.5738888888888889, 'log_fitness': 0.9037974683544304, 'percentage_of_fitting_traces': 52.142857142857146}
[*] Calculating Model Precision...


replaying log with TBR, completed traces ::   0%|          | 0/20 [00:00<?, ?it/s]

    - Precision Score: 0.8564

[SUCCESS] Phase 4 Conformance Audit completed successfully.


In [11]:
import pandas as pd

# 1. Load the original raw dataset available in your directory
df = pd.read_csv('MiniProjet1_SOC_Authentification.csv')

# 2. Parse timestamps and sort chronologically by case
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', dayfirst=True)
df = df.sort_values(by=['incident_id', 'timestamp'])

# 3. Calculate time differences between consecutive events within each case
df['next_timestamp'] = df.groupby('incident_id')['timestamp'].shift(-1)
df['next_activity'] = df.groupby('incident_id')['activity'].shift(-1)
df['transition_duration_mins'] = (df['next_timestamp'] - df['timestamp']).dt.total_seconds() / 60.0

# 4. Analyze average bottlenecks per transition step
bottlenecks = df.groupby(['activity', 'next_activity'])['transition_duration_mins'].mean().reset_index()
bottlenecks = bottlenecks.dropna().sort_values(by='transition_duration_mins', ascending=False)

print("[*] Top Transition Bottlenecks (Average Waiting/Execution Time in Minutes):")
print(bottlenecks.to_string(index=False))

[*] Top Transition Bottlenecks (Average Waiting/Execution Time in Minutes):
               activity      next_activity  transition_duration_mins
             SOC_Triage   Account_Disabled                 11.400000
           Login_Failed      Login_Success                 11.210526
     SIEM_Alert_Created         SOC_Triage                 11.144578
          Login_Success     Session_Closed                 10.824561
             IP_Blocked   Account_Disabled                 10.782051
             SOC_Triage         SOC_Triage                 10.750000
Multiple_Login_Failures SIEM_Alert_Created                 10.710843
       Account_Disabled     Password_Reset                 10.641026
             SOC_Triage    IP_Investigated                 10.418919
        IP_Investigated         IP_Blocked                 10.135135
         Password_Reset    Incident_Closed                  9.910256
       Account_Disabled    Incident_Closed                  7.200000
             SOC_Triage    

In [12]:
# ==========================================
# CELL 6: PHASE 5 - ADVANCED PERFORMANCE & REWORK ANALYSIS
# ==========================================
import pandas as pd
import numpy as np

# 1. Load and prepare data (using the raw file to avoid path errors)
df = pd.read_csv('MiniProjet1_SOC_Authentification.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', dayfirst=True)
df = df.sort_values(by=['incident_id', 'timestamp'])

# 2. Aggregate case-level metrics
cases = df.groupby('incident_id').agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    risk_level=('risk_level', 'first'),
    department=('department', 'first'),
    scenario_label=('scenario_label', 'first'),
    event_count=('activity', 'count'),
    unique_activities=('activity', 'nunique')
).reset_index()

# Calculate Cycle Time in minutes
cases['cycle_time_mins'] = (cases['end_time'] - cases['start_time']).dt.total_seconds() / 60.0

# Detect Rework (If a case has more events than unique activities, a step was repeated)
cases['has_rework'] = cases['event_count'] > cases['unique_activities']

# 3. Print the required Phase 5 analytical answers
print("[*] --- PHASE 5: CYCLE TIME & PERFORMANCE ANALYSIS --- [*]\n")

print("1. Top 3 Longest Incidents (Goulots d'étranglement majeurs):")
print(cases.sort_values(by='cycle_time_mins', ascending=False)[['incident_id', 'cycle_time_mins', 'scenario_label']].head(3).to_string(index=False))

print("\n2. Impact of Rework on Cycle Time:")
print(cases.groupby('has_rework')['cycle_time_mins'].mean().reset_index().rename(columns={'has_rework': 'Contains Rework', 'cycle_time_mins': 'Avg Duration (mins)'}).to_string(index=False))

print("\n3. Performance by Risk Level (Are high risks treated faster?):")
print(cases.groupby('risk_level')['cycle_time_mins'].mean().reset_index().to_string(index=False))

print("\n4. Performance by Attack Scenario:")
print(cases.groupby('scenario_label')['cycle_time_mins'].mean().reset_index().to_string(index=False))

[*] --- PHASE 5: CYCLE TIME & PERFORMANCE ANALYSIS --- [*]

1. Top 3 Longest Incidents (Goulots d'étranglement majeurs):
incident_id  cycle_time_mins     scenario_label
  AUTH-0042            106.0         BruteForce
  AUTH-0113            103.0 CredentialStuffing
  AUTH-0112             98.0 CredentialStuffing

2. Impact of Rework on Cycle Time:
 Contains Rework  Avg Duration (mins)
           False            51.096296
            True            59.600000

3. Performance by Risk Level (Are high risks treated faster?):
risk_level  cycle_time_mins
      High        53.666667
       Low        50.500000
    Medium        49.980000

4. Performance by Attack Scenario:
    scenario_label  cycle_time_mins
            Benign        22.035088
        BruteForce        74.885714
CompromisedAccount        71.764706
CredentialStuffing        67.709677


In [13]:
# ==========================================
# CELL 7: PHASE 6 - CYBERSECURITY THREAT ANALYSIS
# ==========================================
import pandas as pd

df = pd.read_csv('MiniProjet1_SOC_Authentification.csv')
df = df.sort_values(by=['incident_id', 'timestamp'])

# Group activities into an ordered list (trace) per incident
traces = df.groupby('incident_id').agg(
    trace=('activity', list),
    scenario=('scenario_label', 'first'),
    risk=('risk_level', 'first')
).reset_index()

# 1. Identify specific security deviations requested by the professor
traces['missing_ip_investigation'] = traces['trace'].apply(lambda x: 'IP_Investigated' not in x and 'Incident_Closed' in x)
traces['rework_triage'] = traces['trace'].apply(lambda x: x.count('SOC_Triage') > 1)

print("[*] --- PHASE 6: CYBERSECURITY COMPLIANCE ANALYSIS --- [*]\n")

print("1. Critical Security Gaps (Incidents closed WITHOUT IP Investigation):")
missing_ip_cases = traces[traces['missing_ip_investigation']]
print(f"   -> Found {len(missing_ip_cases)} cases skipping IP investigation.")
if len(missing_ip_cases) > 0:
    print(missing_ip_cases[['incident_id', 'scenario', 'risk']].head().to_string(index=False))

print("\n2. Operational Waste (Incidents with duplicated SOC_Triage):")
rework_cases = traces[traces['rework_triage']]
print(f"   -> Found {len(rework_cases)} cases with redundant triage.")

print("\n3. Scenario Execution Patterns (BruteForce vs CredentialStuffing):")
scenario_counts = traces.groupby('scenario').size().reset_index(name='Total Incidents')
print(scenario_counts.to_string(index=False))

[*] --- PHASE 6: CYBERSECURITY COMPLIANCE ANALYSIS --- [*]

1. Critical Security Gaps (Incidents closed WITHOUT IP Investigation):
   -> Found 9 cases skipping IP investigation.
incident_id           scenario   risk
  AUTH-0024 CredentialStuffing Medium
  AUTH-0039 CredentialStuffing   High
  AUTH-0047         BruteForce Medium
  AUTH-0058         BruteForce   High
  AUTH-0070 CredentialStuffing   High

2. Operational Waste (Incidents with duplicated SOC_Triage):
   -> Found 4 cases with redundant triage.

3. Scenario Execution Patterns (BruteForce vs CredentialStuffing):
          scenario  Total Incidents
            Benign               57
        BruteForce               35
CompromisedAccount               17
CredentialStuffing               31


In [14]:
# ==========================================
# CELL 8: PHASE 8 - PREDICTIVE PROCESS MINING (MACHINE LEARNING)
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

# We use the 'cases' dataframe generated in Cell 6
# Goal: Predict if an incident will take longer than 60 minutes (SLA Breach) based on early attributes
cases['SLA_Breached'] = (cases['cycle_time_mins'] > 60).astype(int)

# Select features available at the START of an incident
features = ['risk_level', 'department', 'scenario_label']
X = pd.get_dummies(cases[features]) # One-hot encode categorical variables
y = cases['SLA_Breached']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate predictions
y_pred = clf.predict(X_test)

print("[*] --- PHASE 8: PREDICTIVE MACHINE LEARNING MODEL --- [*]\n")
print("Task: Predicting if an incident will breach the 60-minute resolution SLA.")
print("Algorithm: Random Forest Classifier")
print("\nModel Accuracy & Metrics:")
print(classification_report(y_test, y_pred, target_names=['SLA Met (<=60m)', 'SLA Breached (>60m)']))

# Show feature importance to explain what drives long resolution times
importance = pd.DataFrame({'Feature': X.columns, 'Importance': clf.feature_importances_})
importance = importance.sort_values(by='Importance', ascending=False).head(3)
print("\nTop 3 Factors driving Incident Duration:")
print(importance.to_string(index=False))

[*] --- PHASE 8: PREDICTIVE MACHINE LEARNING MODEL --- [*]

Task: Predicting if an incident will breach the 60-minute resolution SLA.
Algorithm: Random Forest Classifier

Model Accuracy & Metrics:
                     precision    recall  f1-score   support

    SLA Met (<=60m)       0.90      0.78      0.84        23
SLA Breached (>60m)       0.77      0.89      0.83        19

           accuracy                           0.83        42
          macro avg       0.84      0.84      0.83        42
       weighted avg       0.84      0.83      0.83        42


Top 3 Factors driving Incident Duration:
                          Feature  Importance
            scenario_label_Benign    0.417531
        scenario_label_BruteForce    0.145329
scenario_label_CredentialStuffing    0.078927
